# Generating manuscript figures and tables

### Imports and Functions

In [2]:
# basic imports
import os
import warnings
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm

# plotting
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
# import ptitprince as pt

# statistics
from sklearn.metrics import roc_curve, precision_recall_curve, auc
from tableone import TableOne
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy.stats import chi2, spearmanr, friedmanchisquare, wilcoxon, mannwhitneyu
from statsmodels.stats.anova import anova_lm
import statsmodels.stats.anova as sms_anova
from itertools import combinations
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

# ── Style ────────────────────────────────────────────────────────────────────
plt.style.use("seaborn-v0_8-ticks")
sns.set_context("paper", font_scale=1.6)
plt.rcParams.update({"font.family": "sans-serif", "figure.dpi": 300})

# ── Constants ─────────────────────────────────────────────────────────────────
# Canonical montage order (mirrors Supp Table 1)
SUPP_TABLE1_ORDER = [
    "full",                       # Full 16-channel bipolar
    "epiminder_2",                # C3-P3, C4-P4   (Epiminder simulated via interpolation)
    "uneeg_diag_left_front",      # F3-T3
    "uneeg_diag_right_front",     # F4-T4
    "uneeg_diag_bilateral_front", # F3-T3, F4-T4
    "uneeg_left_front",           # F7-T3
    "uneeg_right_front",          # F8-T4
    "uneeg_bilateral_front2",     # F7-T3, F8-T4
    "uneeg_vert_left",            # C3-T3
    "uneeg_vert_right",           # C4-T4
    "uneeg_vert_bilateral",       # C3-T3, C4-T4
    "uneeg_diag_left_back",       # P3-T3
    "uneeg_diag_right_back",      # P4-T4
    "uneeg_diag_bilateral_back",  # P3-T3, P4-T4
    "uneeg_left_back",            # T3-T5
    "uneeg_right_back",           # T4-T6
    "uneeg_bilateral_back2",      # T3-T5, T4-T6
    "ceribell"
]

# Human-readable labels — channel-pair names only
MONTAGE_LABELS = {
    "full":                       "Full",
    "epiminder_2":                "C3-P3, C4-P4",
    "uneeg_diag_left_front":      "F3-T3",
    "uneeg_diag_right_front":     "F4-T4",
    "uneeg_diag_bilateral_front": "F3-T3, F4-T4",
    "uneeg_left_front":           "F7-T3",
    "uneeg_right_front":          "F8-T4",
    "uneeg_bilateral_front2":     "F7-T3, F8-T4",
    "uneeg_vert_left":            "C3-T3",
    "uneeg_vert_right":           "C4-T4",
    "uneeg_vert_bilateral":       "C3-T3, C4-T4",
    "uneeg_diag_left_back":       "P3-T3",
    "uneeg_diag_right_back":      "P4-T4",
    "uneeg_diag_bilateral_back":  "P3-T3, P4-T4",
    "uneeg_left_back":            "T3-T5",
    "uneeg_right_back":           "T4-T6",
    "uneeg_bilateral_back2":      "T3-T5, T4-T6",
    "ceribell":                   "Circumferential",
}

# Selected montages for main figures (Fig 2, Fig 3)
SELECTED_MONTAGES = [
    "full",
    "uneeg_diag_bilateral_front",
    "uneeg_diag_left_front",
    "uneeg_diag_right_front",
    "epiminder_2",
]

FULL_MONTAGES = list(MONTAGE_LABELS.keys())

colors = ["#cecece", "#082a54", "#e02b35", "#F47C6D", "#368899"]
MONTAGE_PALETTE = {k:v for k,v in zip(SELECTED_MONTAGES, colors)}

patient_info = "/Volumes/users/haoershi/emu_dataset/dataset_admission_info.csv"
seizure_data = "/Volumes/users/haoershi/emu_dataset/ueoeec_seizure_details.csv"

# ── Helpers ───────────────────────────────────────────────────────────────────

def save_figure(out_dir, basename):
    """
    Save the current matplotlib figure as both PNG (300 dpi) and SVG
    so that figures can be further edited in vector graphics tools.

    Parameters
    ----------
    out_dir  : str   Output directory (must already exist).
    basename : str   Filename *without* extension, e.g. 'Figure2_MultiModel_strip'.

    Returns
    -------
    png_path : str   Path to the saved PNG file.
    """
    png_path = os.path.join(out_dir, f"{basename}.png")
    svg_path = os.path.join(out_dir, f"{basename}.svg")
    plt.savefig(png_path, bbox_inches="tight", dpi=300)
    plt.savefig(svg_path, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {png_path}")
    print(f"  Saved: {svg_path}")
    return png_path

def load_all_metrics(results_dir, setting_folder):
    """Load patient_metrics.csv and segment_metrics.csv for every montage × model subfolder."""
    base = os.path.join(results_dir, "metrics", setting_folder)
    if not os.path.exists(base):
        print(f"[WARN] Path not found: {base}")
        return pd.DataFrame(), pd.DataFrame()

    montages = [d for d in os.listdir(base) if os.path.isdir(os.path.join(base, d))]
    p_dfs, s_dfs = [], []

    for m in tqdm(montages, desc=f"Loading {setting_folder}"):
        m_path = os.path.join(base, m)

        # --- patient metrics ---
        p_file = os.path.join(m_path, "patient_metrics.csv")
        if os.path.exists(p_file):
            try:
                df = pd.read_csv(p_file)
                if "patient_id" not in df.columns and "Unnamed: 0" in df.columns:
                    df = df.rename(columns={"Unnamed: 0": "patient_id"})
                # Normalise AUROC / AUPRC column names to canonical 'auroc' / 'auprc'.
                # The actual pipeline writes them as 'auroc_sample' / 'auprc_sample';
                # aliases cover any other naming conventions used by upstream code.
                _AUROC_ALIASES = ["auroc_sample", "roc_auc", "auc_roc", "auc",
                                  "AUROC", "auroc_score"]
                _AUPRC_ALIASES = ["auprc_sample", "pr_auc", "auc_pr", "prauc",
                                  "AUPRC", "auprc_score", "average_precision"]
                if "auroc" not in df.columns:
                    for alias in _AUROC_ALIASES:
                        if alias in df.columns:
                            df = df.rename(columns={alias: "auroc"})
                            break
                if "auprc" not in df.columns:
                    for alias in _AUPRC_ALIASES:
                        if alias in df.columns:
                            df = df.rename(columns={alias: "auprc"})
                            break
                df["montage"] = m
                p_dfs.append(df)
            except Exception as e:
                print(f"  Error {p_file}: {e}")

        # --- segment / seizure metrics ---
        s_file = os.path.join(m_path, "segment_metrics.csv")
        if os.path.exists(s_file):
            try:
                df = pd.read_csv(s_file)
                if "Unnamed: 0" in df.columns:
                    df = df.rename(columns={"Unnamed: 0": "event_id"})
                if "recall_event" in df.columns:
                    df["detected_binary"] = (df["recall_event"] > 0).astype(int)
                df["montage"] = m
                s_dfs.append(df)
            except Exception as e:
                print(f"  Error {s_file}: {e}")

    df_p = pd.concat(p_dfs, ignore_index=True) if p_dfs else pd.DataFrame()
    df_s = pd.concat(s_dfs, ignore_index=True) if s_dfs else pd.DataFrame()
    return df_p, df_s


def load_multi_model_metrics(results_dir_map, setting_map):
    """
    Load metrics for multiple models.

    Parameters
    ----------
    results_dir_map : dict  {model_name: results_dir}
    setting_map     : dict  {model_name: setting_folder}

    Returns
    -------
    df_p, df_s  – combined DataFrames with 'model' column
    """
    all_p, all_s = [], []
    for model_name, rdir in results_dir_map.items():
        setting = setting_map.get(model_name, "thres_optimal")
        print(f"\n=== Loading {model_name} from {rdir} / {setting} ===")
        dp, ds = load_all_metrics(rdir, setting)
        if not dp.empty:
            dp["model"] = model_name
            all_p.append(dp)
        if not ds.empty:
            ds["model"] = model_name
            all_s.append(ds)
        if 'admission_id' not in dp.columns:
            dp.columns = ['admission_id'] + list(dp.columns[1:])

    df_p = pd.concat(all_p, ignore_index=True) if all_p else pd.DataFrame()
    df_s = pd.concat(all_s, ignore_index=True) if all_s else pd.DataFrame()
    return df_p, df_s


def load_metadata(patient_info_path, seizure_details_path):
    """
    Load dataset_admission_info.csv and (optionally) seizure details.

    dataset_admission_info.csv columns used:
        admission_id   – primary join key (e.g. EMU0878)
        patient_id     – numeric patient identifier
        laterality     – left / right / bilateral  (Title-cased on load)
        location       – temporal / frontal / multifocal  (Title-cased)
        epilepsy_type  – focal / gen / mixed  ('gen' -> 'Generalized')
    """
    p_info, s_info = pd.DataFrame(), pd.DataFrame()

    if os.path.exists(patient_info_path):
        p_info = pd.read_csv(
            patient_info_path,
            dtype={"admission_id": str, "patient_id": str},
        )
        for col in ["laterality", "location", "epilepsy_type", "gender"]:
            if col in p_info.columns:
                p_info[col] = p_info[col].astype(str).str.strip().str.title()
        # 'Gen' is stored as abbreviation in the CSV
        p_info["epilepsy_type"] = p_info["epilepsy_type"].replace({"Gen": "Generalized"})
        print(f"Loaded patient info: {len(p_info)} admissions from {patient_info_path}")
    else:
        print(f"[WARN] Patient info not found: {patient_info_path}")

    if os.path.exists(seizure_details_path):
        s_info = pd.read_csv(seizure_details_path)
        if "off" in s_info.columns and "ueoeec" in s_info.columns:
            s_info["duration"] = s_info["off"] - s_info["ueoeec"]
        print(f"Loaded seizure details: {len(s_info)} rows from {seizure_details_path}")
    else:
        print(f"[WARN] Seizure details not found: {seizure_details_path}")

    return p_info, s_info

# ── Helpers ───────────────────────────────────────────────────────────────────

def bootstrap_ci(data, n_boot=1000, ci=95, seed=42):
    data = np.asarray(data, dtype=float)
    data = data[~np.isnan(data)]
    if len(data) == 0:
        return np.nan, np.array([[np.nan], [np.nan]])
    if len(data) < 2:
        return float(data[0]), np.array([[0.0], [0.0]])
    rng = np.random.default_rng(seed)
    boot = [rng.choice(data, size=len(data), replace=True).mean() for _ in range(n_boot)]
    mean = np.mean(data)
    lo = np.percentile(boot, (100 - ci) / 2)
    hi = np.percentile(boot, 100 - (100 - ci) / 2)
    return mean, np.array([[mean - lo], [hi - mean]])

def label(m):
    return MONTAGE_LABELS.get(m, m)

def ordered_montages(df, montages_wanted, col="montage"):
    avail = df[col].unique()
    return [m for m in montages_wanted if m in avail]

def _get_models(df, models_wanted, col="model"):
    if "model" in df.columns:
        return [m for m in models_wanted if m in df["model"].unique().tolist()]
    return ["(single model)"]

def load_full_prob(results_dir_map, setting_map):
    """
    Load metrics for multiple models.

    Parameters
    ----------
    results_dir_map : dict  {model_name: results_dir}
    setting_map     : dict  {model_name: setting_folder}

    Returns
    -------
    df_p, df_s  – combined DataFrames with 'model' column
    """
    all_data = {}
    for model_name, rdir in results_dir_map.items():
        pred_dir = os.path.join(rdir,'pred',setting_map[model_name])
        print(f"\n=== Loading {model_name} from {pred_dir} ===")
        montages = os.listdir(pred_dir)
        use_montages = [m for m in SELECTED_MONTAGES if m in montages]
        model_data = []
        for m in use_montages:
            prob_files = os.listdir(os.path.join(pred_dir, m))
            for f in tqdm(prob_files):
                tmp = pd.read_csv(os.path.join(pred_dir,m,f))
                tmp = tmp[['sz_prob','label']] if 'sz_prob' in tmp.columns else tmp[['LPD','label']].rename(columns={'LPD':'sz_prob'}) 
                tmp['event_id'] = f[:-4]
                tmp['montage'] = m
                model_data.append(tmp)
        model_data = pd.concat(model_data)
        model_data['patient'] = model_data['event_id'].apply(lambda x: x.split('_')[0])
        all_data[model_name] = model_data
    return all_data

In [ ]:
model_dirs = ['sparcnet_results',
                'ndd_results',
                'svm_results']
model_names = ['SPaRCNet', 'NDD', 'SVM']
results_dir_map = dict(zip(model_names, model_dirs))
# threshold setting
setting_folder = 'thres_optimal_f1'
settings = ['thres_optimal_f1', 'thres_optimal_f1', 'thres_optimal_f1_avg'] #[setting_folder] * len(model_names)
setting_map = dict(zip(model_names, settings))
df_p, df_s = load_multi_model_metrics(results_dir_map, setting_map)     
if df_p.empty:
    raise SystemExit("No patient metrics loaded. Check your directory structure.")
out_dir = 'manuscript'
os.makedirs(out_dir, exist_ok=True)
print(f"\nOutput directory: {out_dir}")
print(f"Patient rows loaded: {len(df_p)}")
print(f"Seizure rows loaded: {len(df_s)}")

# ── Load metadata ───────────────────────────────────────────────────────────
p_info, s_info = load_metadata(patient_info, seizure_data)

# ── Load probability data ───────────────────────────────────────────────────
prob_data = load_full_prob(results_dir_map, setting_map)

### Table 1

In [15]:
patient_info = pd.read_csv('../emu_dataset/dataset_patient_info.csv')

In [21]:
var_labels = {'age_1st_adm':'Age at 1st Admission',
              'gender':'Sex',
              'num_sz_event':'Number of Seizures',
              'num_iic_segment':'Number of IIC Segments',
              'avg_sz_dura':'Seizure Duration (sec)',
              'epilepsy_type':'Epilepsy Type',
              'laterality':'Epilepsy Laterality',
              'location':'Epilepsy Localization'}
table1 = TableOne(patient_info, 
                  columns=['age_1st_adm','gender', 'num_sz_event', 'num_iic_segment', 'avg_sz_dura', 'epilepsy_type', 'laterality', 'location'], 
                  categorical=['gender', 'laterality', 'location', 'epilepsy_type'],
                  nonnormal=['num_sz_event','num_iic_segment'], rename=var_labels, missing=False)
table1.to_csv(f'{out_dir}/table1.csv')

In [22]:
table1 = TableOne(patient_info[patient_info['epilepsy_type'].isin(['focal', 'mixed'])], 
                  columns=['laterality', 'location'], 
                  categorical=['laterality', 'location'], rename=var_labels, missing=False)
table1.to_csv(f'{out_dir}/table1_focal_mixed.csv')

### Figure 2

In [8]:
# ── Figure 2 – Stripplot / Violin + Mean CI, 3 metrics × 3 models ────────────
def generate_figure2_multi(df_p, out_dir, style="strip", suffix=''):
    """
    Figure 2: 5 selected montages × 3 models × 3 metrics.
    style = 'strip' (jittered dots) or 'violin'
    """
    models = _get_models(df_p, SELECTED_MODELS)
    montages = ordered_montages(df_p, SELECTED_MONTAGES)
    if not montages:
        print("[WARN] No selected montages found for Fig 2. Skipping.")
        return

    metrics = [
        ("f1_event",     "F1 Score",            (-0.03, 1.03)),
        ("recall_event", "Sensitivity",          (-0.03, 1.03)),
        ("fp",           "False Alarms / Hour",  None),
    ]
    available_metrics = [(c, n, yl) for c, n, yl in metrics if c in df_p.columns]
    if not available_metrics:
        print("[WARN] No metric columns found in patient DataFrame. Skipping Fig 2.")
        return

    n_metrics = len(available_metrics)
    n_models  = len(models)

    fig, axes = plt.subplots(
        n_models, n_metrics,
        figsize=(5 * n_metrics, 4 * n_models+1),
        squeeze=False,
        sharex=True,
        sharey="col",
    )

    for row_i, model in enumerate(models):
        sub = df_p[df_p["model"] == model] if "model" in df_p.columns else df_p

        for col_i, (col, name, ylim) in enumerate(available_metrics):
            ax = axes[row_i][col_i]
            if col not in sub.columns:
                ax.set_visible(False)
                continue

            plot_data = sub[sub["montage"].isin(montages)].copy()
            plot_data["montage_label"] = plot_data["montage"].map(label)
            mont_labels = [label(m) for m in montages]

            if style == "violin":
                sns.violinplot(
                    data=plot_data, x="montage_label", y=col, order=mont_labels,
                    palette={label(m): MONTAGE_PALETTE[m] for m in montages},
                    inner=None, cut=0, linewidth=0.8, alpha=0.6, ax=ax,
                )
            elif style == 'strip':  # strip
                sns.stripplot(
                    data=plot_data, x="montage_label", y=col, order=mont_labels,
                    palette={label(m): MONTAGE_PALETTE[m] for m in montages},
                    jitter=0.2, size=3, alpha=0.3, zorder=1, ax=ax,
                )
        
                sns.pointplot(
                        data=plot_data, x="montage_label", y=col, order=mont_labels,
                        # palette={label(m): mont_color[m] for m in montages},
                        estimator='mean',
                        dodge=False, linestyle="none", errorbar=("ci", 95),
                        marker="_", markersize=25, markeredgewidth=3,zorder=10,errwidth=1,color='black', ax=ax
                    )
            elif style == 'box':
                sns.boxplot(
                        data=plot_data, x="montage_label", y=col, order=mont_labels,
                        palette={label(m): MONTAGE_PALETTE[m] for m in montages},#sns.color_palette("Paired"),#
                        notch=False, showcaps=True,showfliers=False,width=0.5,
                        # boxprops={'edgeco'}
                        medianprops={"color": "black", "linewidth": 2},zorder=0,
                        ax=ax
                    )
                
                for patch in ax.patches:
                    color = patch.get_facecolor()
                    patch.set_facecolor((*color[:3], 0.5)) # transparent
                    # patch.set_edgecolor(color)
                    patch.set_linewidth(1)
                # for i, line in enumerate(ax.lines):
                #     color = ax.patches[i // 6].get_edgecolor()  # each box has ~6 lines
                #     line.set_color(color)
                #     line.set_linewidth(2)

                sns.stripplot(
                    data=plot_data, x="montage_label", y=col, order=mont_labels,
                    palette={label(m): MONTAGE_PALETTE[m] for m in montages},#sns.color_palette("Paired"),#
                    jitter=0.15, size=4, alpha=0.5, zorder=1, ax=ax,
                )
        
                
            elif style == 'raincloud':
                pt.RainCloud(
                    x="montage_label", y=col, data=plot_data, order=mont_labels, hue_order=mont_labels,
                    palette=[MONTAGE_PALETTE[m] for m in montages], #sns.color_palette("viridis"),#
                    bw=0.2, width_viol=0.6, ax=ax, orient="v", alpha=0.8, pointplot=True, move=0.25, offset=0.1, jitter=0.12,
                    box_showfliers=False, rain_alpha=0.5
                )
            if ylim:
                ax.set_ylim(ylim)
            ax.set_title(f"{name}" if row_i == 0 else "", fontweight="bold", pad=8)
            ax.set_ylabel(model if col_i == 0 else "", fontweight="bold")
            ax.set_xlabel("")
            ax.set_xticklabels(mont_labels, rotation=40, ha="right", fontsize=12)
            if col_i ==1:
                ax.set_yticklabels([])
                ax.yaxis.set_tick_params(labelleft=False)
            # ax.yaxis.grid(True, linestyle="--", alpha=0.5)

    plt.suptitle("Seizure Detection Performance – Selected Montages", fontsize=15, y=1.01)
    sns.despine()
    plt.tight_layout()
    save_figure(out_dir, f"figure2_multimodel_{style}{suffix}")

def generate_figure2_f1(df_p, out_dir, style="strip", suffix=''):
    """
    Figure 2: 5 selected montages × 3 models × 3 metrics.
    style = 'strip' (jittered dots) or 'violin'
    """
    models = _get_models(df_p, SELECTED_MODELS)
    montages = ordered_montages(df_p, SELECTED_MONTAGES)
    if not montages:
        print("[WARN] No selected montages found for Fig 2. Skipping.")
        return

    metrics = [
        ("f1_event",     "F1 Score",            (-0.03, 1.03))
    ]
    available_metrics = [(c, n, yl) for c, n, yl in metrics if c in df_p.columns]
    col, name, ylim = available_metrics[0]
    n_metrics = len(available_metrics)
    n_models  = len(models)

    fig, axes = plt.subplots(n_metrics, n_models,
        figsize=(4 * n_models, 5*n_metrics), squeeze=True
    )
    if (n_metrics == 1) & (n_models == 1):
        axes = [axes]

    df_p["montage_label"] = df_p["montage"].map(label)
    mont_labels = [label(m) for m in montages]
    for row_i, model in enumerate(models):
        sub = df_p[df_p["model"] == model] if "model" in df_p.columns else df_p
        sub = sub[sub["montage"].isin(montages)].copy()
        ax = axes[row_i]
        if style == "violin":
            sns.violinplot(
                data=sub, x="montage_label", y=col, order = mont_labels,
                palette={label(m): MONTAGE_PALETTE[m] for m in montages},
                inner=None, cut=0, linewidth=0.8, alpha=0.6, ax=ax,
            )
        elif style == 'strip':  # strip
            sns.stripplot(
                data=sub, x="montage_label", y=col, order = mont_labels,
                palette={label(m): MONTAGE_PALETTE[m] for m in montages}, 
                jitter=0.2, size=3, alpha=0.3, zorder=1, ax=ax,
            )
    
            sns.pointplot(
                    data=sub, x="montage_label", y=col, order = mont_labels,
                    # palette={label(m): mont_color[m] for m in montages},
                    estimator='mean',
                    dodge=False, linestyle="none", errorbar=("ci", 95),
                    marker="_", markersize=25, markeredgewidth=3,zorder=10,errwidth=1,color='black', ax=ax
                )
        elif style == 'box':
            sns.boxplot(
                    data=sub, x="montage_label", y=col, order = mont_labels,
                    palette={label(m): MONTAGE_PALETTE[m] for m in montages},#sns.color_palette("Paired"),#
                    notch=False, showcaps=True,showfliers=False,width=0.5,
                    # boxprops={'edgeco'}
                    medianprops={"color": "black", "linewidth": 2},zorder=0,
                    ax=ax
                )
            
            for patch in ax.patches:
                color = patch.get_facecolor()
                patch.set_facecolor((*color[:3], 0.5)) # transparent
                # patch.set_edgecolor(color)
                patch.set_linewidth(1)
            # for i, line in enumerate(ax.lines):
            #     color = ax.patches[i // 6].get_edgecolor()  # each box has ~6 lines
            #     line.set_color(color)
            #     line.set_linewidth(2)
    
            sns.stripplot(
                data=sub, x="montage_label", y=col, order = mont_labels,
                palette={label(m): MONTAGE_PALETTE[m] for m in montages},#sns.color_palette("Paired"),#
                jitter=0.15, size=4, alpha=0.5, zorder=1, ax=ax,
            )
    
            
        elif style == 'raincloud':
            pt.RainCloud(
                    x="montage_label", y=col, data=sub, order=mont_labels, hue_order=mont_labels,
                    palette=[MONTAGE_PALETTE[m] for m in montages], #sns.color_palette("viridis"),#
                    bw=0.2, width_viol=0.6, ax=ax, orient="v", alpha=0.7, pointplot=True, move=0.25, offset=0.1, jitter=0.12,
                    box_showfliers=False, rain_alpha=0.5
                )
        if ylim:
            ax.set_ylim(ylim)
        ax.set_title(f"{model}", fontweight="bold", pad=8)
        ax.set_ylabel(name if row_i == 0 else "", fontweight="bold")
        ax.set_xlabel("")
        ax.set_xticklabels(mont_labels, rotation=40, ha='right', fontsize=12)
        if row_i != 0:
            ax.set_yticklabels([])
            ax.yaxis.set_tick_params(labelleft=False)
        # ax.yaxis.grid(True, linestyle="--", alpha=0.5)

    plt.suptitle("Seizure Detection Performance – Selected Montages", fontsize=15, y=1.01)
    sns.despine()
    plt.tight_layout()
    save_figure(out_dir, f"figure2_f1_{style}{suffix}")



In [9]:
SELECTED_MONTAGES = [
    "full",
    "uneeg_diag_bilateral_front",
    "uneeg_diag_left_front",
    "uneeg_diag_right_front",
    "epiminder_2",
]
SELECTED_MODELS = ['SPaRCNet','NDD','SVM']

# ── Generate all outputs ────────────────────────────────────────────────────
print("\n=== Figure 2: Strip/Violin + Mean CI (multi-model) ===")
generate_figure2_multi(df_p, out_dir, style='box')
generate_figure2_f1(df_p, out_dir, style='box')


=== Figure 2: Strip/Violin + Mean CI (multi-model) ===
  Saved: manuscript/figure2_multimodel_box.png
  Saved: manuscript/figure2_multimodel_box.svg
  Saved: manuscript/figure2_f1_box.png
  Saved: manuscript/figure2_f1_box.svg


### Figure 3

In [10]:
# ── Figure 3 – Patient-level stratified heatmap ───────────────────────────────
def generate_figure3_patient(df_p, p_info, out_dir, fig_name):
    """
    Figure 3: Sensitivity heatmap stratified by patient-level variables from
    dataset_admission_info.csv:
      - Epilepsy type     (Focal / Generalized / Mixed)
      - Epilepsy laterality (Left / Right / Bilateral)
      - Epilepsy location  (Temporal / Frontal / Multifocal)
    Join key: admission_id (EMU-format string).
    Rows = selected montages sorted by mean sensitivity descending.
    One model panel per row; one stratification panel per column.
    """
    print("Generating Figure 3 – Patient-level stratified heatmap...")
    if df_p.empty or p_info.empty:
        print("  [WARN] Missing data. Skipping Fig 3.")
        return

    metric_col = "recall_event" if "recall_event" in df_p.columns else None
    if metric_col is None:
        print("  [WARN] recall_event not found. Skipping Fig 3.")
        return

    # ── Join on admission_id ──────────────────────────────────────────────────
    df_p = df_p.copy()
    # patient_metrics.csv may index by admission_id or patient_id
    if "admission_id" in df_p.columns:
        df_p["admission_id"] = df_p["admission_id"].astype(str).str.upper().str.strip()
    elif "patient_id" in df_p.columns:
        df_p = df_p.rename(columns={"patient_id": "admission_id"})
        df_p["admission_id"] = df_p["admission_id"].astype(str).str.upper().str.strip()

    p_info = p_info.copy()
    p_info["admission_id"] = p_info["admission_id"].astype(str).str.upper().str.strip()

    merged = df_p.merge(p_info, on="admission_id", how="left")
    n_matched = merged["laterality"].notna().sum()
    print(f"  Merged: {len(merged)} rows, {n_matched} with metadata matched.")

    montages  = ordered_montages(merged, SELECTED_MONTAGES)
    sub       = merged[merged["montage"].isin(montages)].copy()

    models = _get_models(sub, SELECTED_MODELS)

    # ── Build stratification specs ────────────────────────────────────────────
    # Each entry: (panel_title, categories_in_order, col_in_df, display_name_map)
    STRAT_SPECS = [
        (
            "Epilepsy Type",
            ["Focal", "Generalized"],
            "epilepsy_type",
            {},
        ),
        (
            "Laterality",
            ["Left", "Right", "Bilateral"],
            "laterality",
            {},
        ),
        (
            "Location",
            ["Temporal", "Frontal"],
            "location",
            {},
        ),
    ]

    # Filter specs to columns actually present after merge
    valid_specs = [s for s in STRAT_SPECS if s[2] in sub.columns]
    if not valid_specs:
        print("  [WARN] None of the stratification columns found after merge. Skipping Fig 3.")
        return

    n_strat  = len(valid_specs)
    n_models = len(models)

    # ── Shared renderer: build and save one stratified heatmap figure ─────────
    def _render_fig3(data_col, cbar_label, suptitle_text, out_basename):
        """Render Figure-3-style grid for a given metric column."""
        if data_col not in sub.columns:
            print(f"  [WARN] Column '{data_col}' not found – skipping {out_basename}.")
            return

        # Sort montages by overall mean of *this* metric
        _mont_order = montages
        # (
        #     sub.groupby("montage")[data_col].mean()
        #     .reindex(montages).sort_values(ascending=False).index.tolist()
        # )
        _mont_labels = [label(m) for m in _mont_order]

        fig, axes = plt.subplots(
            n_models, n_strat,
            figsize=(3.5 * n_strat, min(max(2, len(_mont_order) * 0.25 + 1) * n_models, 12)),
            squeeze=False,
        )
        tmp = sub[(sub['montage'].isin(_mont_order))&sub['model'].isin(models)]
        tmp_means = tmp.groupby(['model','montage'])[data_col].mean()
        vmin, vmax = np.floor(tmp_means.min()*10)/10, np.ceil(tmp_means.max()*10)/10#sub[sub['montage'].isin(_mont_order)][metric_col].min(), sub[sub['montage'].isin(_mont_order)][metric_col].max()

        colors = ["#ffffff","#368899"] 
        cmap = LinearSegmentedColormap.from_list("custom_cmap", colors, N=256)
        for row_i, model in enumerate(models):
            model_sub = sub[sub["model"] == model] if "model" in sub.columns else sub

            for col_i, (title, cats, col, _) in enumerate(valid_specs):
                ax = axes[row_i][col_i]

                ref_mont = _mont_order[0] if _mont_order else model_sub["montage"].iloc[0]
                ref_sub  = model_sub[model_sub["montage"] == ref_mont]

                heat_rows = {}
                for cat in cats:
                    cat_mask = model_sub[col] == cat
                    n_pat    = (ref_sub[col] == cat).sum()
                    if cat_mask.sum() < 5:
                        continue
                    scores = (
                        model_sub[cat_mask]
                        .groupby("montage")[data_col].mean()
                        .reindex(_mont_order)
                    )
                    heat_rows[f"{cat}\n(n={n_pat})"] = scores.values

                if not heat_rows:
                    ax.set_visible(False)
                    continue

                heat_df = pd.DataFrame(heat_rows, index=_mont_labels)
                # cmap = plt.cm.get_cmap("YlPu")
                sns.heatmap(
                    heat_df, annot=True, fmt=".2f", cmap=cmap,#"YlGnBu_r",#"magma"
                    linewidths=0.4, linecolor="white",
                    cbar=False,
                    ax=ax, vmin=vmin, vmax=vmax,
                    cbar_kws={},
                    annot_kws={"size": 9},
                )
                if row_i == 0:
                    ax.set_title(title, fontweight="bold", fontsize=12, pad=8)
                if col_i == 0:
                    ax.set_ylabel(model, fontweight="bold", fontsize=12)
                    ax.set_yticks(np.arange(len(_mont_labels))+0.5)
                    ax.set_yticklabels(_mont_labels, rotation=0, fontsize=9)
                else:
                    ax.set_ylabel("")
                    ax.set_yticks([])
                    ax.set_yticklabels([])
                ax.set_xlabel("")
                ax.xaxis.tick_top()
                ax.set_xticklabels(ax.get_xticklabels(), fontsize=11)
                ax.tick_params(axis='x', which='both', bottom=False, top=False)
                # ax.set_xticks([])
                if row_i > 0:
                    ax.set_xticklabels([])

        # 🔑 create colorbar axis
        cax = fig.add_axes([0.985, 0.15, 0.015, 0.7])#[0.985, 0.10, 0.015, 0.55]
        
        # use one of the heatmaps to generate colorbar
        sm = plt.cm.ScalarMappable(cmap=cmap)
        sm.set_clim(vmin, vmax)
        
        cbar = fig.colorbar(
            sm,
            cax=cax,
            label=cbar_label
        )
        cbar.outline.set_visible(False)
        cbar.ax.tick_params(labelsize=12)
        cbar.set_ticks([vmin, (vmin+vmax)/2, vmax])

        plt.suptitle(suptitle_text, fontsize=15, y=1.01)
        plt.tight_layout()
        save_figure(out_dir, out_basename)

    # ── Produce Sensitivity version (original) ────────────────────────────────
    _render_fig3(
        data_col="recall_event",
        cbar_label="Mean Sensitivity",
        suptitle_text="Sensitivity – Patient-Level Stratification",
        out_basename=f"{fig_name}_stratified_heatmap_sensitivity",
    )

    # ── Produce F1 Score version (new) ────────────────────────────────────────
    _render_fig3(
        data_col="f1_event",
        cbar_label="Mean F1 Score",
        suptitle_text="F1 Score – Patient-Level Stratification",
        out_basename=f"{fig_name}_stratified_heatmap_f1",
    )

    # ── Produce F1 Score version (new) ────────────────────────────────────────
    _render_fig3(
        data_col="fp",
        cbar_label="False Alarm/h",
        suptitle_text="False Alarm/h – Patient-Level Stratification",
        out_basename=f"{fig_name}_stratified_heatmap_FA",
    )




In [11]:
SELECTED_MONTAGES = [
    "full",
    "uneeg_diag_bilateral_front",
    "uneeg_diag_left_front",
    "uneeg_diag_right_front",
    "epiminder_2",
]
SELECTED_MODELS = ['SPaRCNet', 'NDD']
print("\n=== Figure 3: Patient-level stratified heatmap ===")
generate_figure3_patient(df_p, p_info, out_dir, fig_name = 'figure3')


=== Figure 3: Patient-level stratified heatmap ===
Generating Figure 3 – Patient-level stratified heatmap...
  Merged: 25164 rows, 25164 with metadata matched.
  Saved: manuscript/figure3_stratified_heatmap_sensitivity.png
  Saved: manuscript/figure3_stratified_heatmap_sensitivity.svg
  Saved: manuscript/figure3_stratified_heatmap_f1.png
  Saved: manuscript/figure3_stratified_heatmap_f1.svg
  Saved: manuscript/figure3_stratified_heatmap_FA.png
  Saved: manuscript/figure3_stratified_heatmap_FA.svg


In [12]:
SELECTED_MONTAGES = list(MONTAGE_LABELS.keys())
SELECTED_MODELS = ['SPaRCNet','NDD','SVM']
generate_figure3_patient(df_p, p_info, out_dir, fig_name = 'suppfig4')

Generating Figure 3 – Patient-level stratified heatmap...
  Merged: 25164 rows, 25164 with metadata matched.
  Saved: manuscript/suppfig4_stratified_heatmap_sensitivity.png
  Saved: manuscript/suppfig4_stratified_heatmap_sensitivity.svg
  Saved: manuscript/suppfig4_stratified_heatmap_f1.png
  Saved: manuscript/suppfig4_stratified_heatmap_f1.svg
  Saved: manuscript/suppfig4_stratified_heatmap_FA.png
  Saved: manuscript/suppfig4_stratified_heatmap_FA.svg


### Supp Tables

In [13]:
# ── Tables – Detailed performance per model ───────────────────────────────────
def _mean_sd(series):
    v = series.dropna()
    if len(v) == 0:
        return "—"
    return f"{v.mean():.3f} ± {v.std():.3f}"


def _mean_ci(series, n_boot=1000):
    mv, err = bootstrap_ci(series.values, n_boot=n_boot)
    if np.isnan(mv):
        return "—"
    lo = mv - float(err[0][0])
    hi = mv + float(err[1][0])
    return f"{mv:.3f} [{lo:.3f}–{hi:.3f}]"


def generate_supp_tables(df_p, out_dir):
    """
    Generate Supp Table 2/3/4 (one per model).
    Rows = montages in Supp Table 1 canonical order.
    Cols = AUROC, AUPRC, Sensitivity (recall_event), FA/hr (fp), F1 (f1_event).
    Two summary columns per metric: Mean ± SD and Mean [95% CI].
    """
    print("Generating Supplementary Tables...")

    metrics = [
        ("auroc",        "AUROC"),
        ("auprc",        "AUPRC"),
        ("recall_event", "Sensitivity"),
        ("fp",           "FA / Hour"),
        ("f1_event",     "F1 Score"),
    ]

    models = _get_models(df_p, SELECTED_MODELS)

    for model_idx, model in enumerate(models):
        sub = df_p[df_p["model"] == model] if "model" in df_p.columns else df_p

        all_monts = [m for m in SUPP_TABLE1_ORDER if m in sub["montage"].unique()]
        other     = sorted([m for m in sub["montage"].unique() if m not in SUPP_TABLE1_ORDER])
        all_monts = all_monts + other

        rows = []
        for m in all_monts:
            mdata = sub[sub["montage"] == m]
            row = {
                "Montage": label(m).replace("\n", " "),
                "Group":   MONTAGE_LABELS.get(m, "Other"),
                "N (patients)": len(mdata),
            }
            for col, col_name in metrics:
                if col in mdata.columns:
                    row[f"{col_name} (Mean±SD)"]   = _mean_sd(mdata[col])
                    row[f"{col_name} (Mean [95CI])"] = _mean_ci(mdata[col])
                else:
                    row[f"{col_name} (Mean±SD)"]   = "N/A"
                    row[f"{col_name} (Mean [95CI])"] = "N/A"
            rows.append(row)

        table_df = pd.DataFrame(rows)
        safe_name = model.replace(" ", "_").replace("/", "_")

        # 1. Plain CSV
        csv_path = os.path.join(out_dir, f"supptable_model{model_idx+2}_{safe_name}.csv")
        table_df.to_csv(csv_path, index=False)
        print(f"  Saved CSV: {csv_path}")

        # 2. Styled Excel with group coloring
        try:
            import openpyxl
            from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
            from openpyxl.utils import get_column_letter

            wb = openpyxl.Workbook()
            ws = wb.active
            ws.title = f"{model[:28]}"

            # Header
            header_fill = PatternFill("solid", fgColor="2C3E50")
            header_font = Font(color="FFFFFF", bold=True, size=10)
            thin = Side(style="thin", color="CCCCCC")
            border = Border(left=thin, right=thin, top=thin, bottom=thin)

            for col_i, col_name in enumerate(table_df.columns, 1):
                cell = ws.cell(row=1, column=col_i, value=col_name)
                cell.fill = header_fill
                cell.font = header_font
                cell.alignment = Alignment(horizontal="center", wrap_text=True)
                cell.border = border
                ws.column_dimensions[get_column_letter(col_i)].width = max(14, len(col_name) + 2)

            # Data rows with group coloring
            def hex_to_openpyxl(hex_str):
                return hex_str.lstrip("#")

            group_colors_xl = {
                g: hex_to_openpyxl(c) for g, c in {
                    "Full":               "#D5E8D4",
                    "Central":            "#DAE8FC",
                    "Fronto-temporal":    "#E1D5E7",
                    "Centro-temporal":    "#FFE6CC",
                    "Temporo-parietal":   "#F8CECC",
                    "Posterior temporal": "#FFF2CC",
                    "Other":              "#FFFFFF",
                }.items()
            }

            for row_i, (_, row_data) in enumerate(table_df.iterrows(), 2):
                group = row_data.get("Group", "Other")
                fill_color = group_colors_xl.get(group, "FFFFFF")
                fill = PatternFill("solid", fgColor=fill_color)
                for col_i, value in enumerate(row_data, 1):
                    cell = ws.cell(row=row_i, column=col_i, value=value)
                    cell.fill = fill
                    cell.alignment = Alignment(horizontal="center")
                    cell.border = border
                    cell.font = Font(size=9)

            ws.freeze_panes = "A2"
            ws.row_dimensions[1].height = 32

            xlsx_path = os.path.join(out_dir, f"supptable_model{model_idx+2}_{safe_name}.xlsx")
            wb.save(xlsx_path)
            print(f"  Saved Excel: {xlsx_path}")

        except ImportError:
            print("  [NOTE] openpyxl not installed – skipping Excel output. pip install openpyxl")

In [14]:
SELECTED_MONTAGES = list(MONTAGE_LABELS.keys())
SELECTED_MODELS = ['SPaRCNet','NDD','SVM']
print("\n=== Supplementary Tables (one per model) ===")
generate_supp_tables(df_p, out_dir)

print(f"\n✓ Done. All outputs saved to: {out_dir}")


=== Supplementary Tables (one per model) ===
Generating Supplementary Tables...
  Saved CSV: manuscript/supptable_model2_SPaRCNet.csv
  Saved Excel: manuscript/supptable_model2_SPaRCNet.xlsx
  Saved CSV: manuscript/supptable_model3_NDD.csv
  Saved Excel: manuscript/supptable_model3_NDD.xlsx
  Saved CSV: manuscript/supptable_model4_SVM.csv
  Saved Excel: manuscript/supptable_model4_SVM.xlsx

✓ Done. All outputs saved to: manuscript


### SuppFig2 Samplewise ROC/PRC

In [ ]:
def bootstrap_roc_patient_level(df, y_true_col='label', y_score_col='sz_prob', patient_col='patient', n_bootstrap=1000, seed=42):
    rng = np.random.RandomState(seed)

    patients = df[patient_col].unique()
    n_patients = len(patients)
    
    mean_fpr = np.linspace(0, 1, 200)

    y_true = df[y_true_col].values
    y_score = df[y_score_col].values
    full_fpr, full_tpr, _ = roc_curve(y_true, y_score)
    # Downsample to 200 values by subsampling (not interpolation)
    if len(full_fpr) > 1000:
        indices = np.round(np.linspace(0, len(full_fpr) - 1, 1000)).astype(int)
        full_fpr = full_fpr[indices]
        full_tpr = full_tpr[indices]
    roc_auc = auc(full_fpr, full_tpr)

    tprs = []
    aucs = []

    for _ in tqdm(range(n_bootstrap)):
        # 🔹 resample patients (with replacement)
        sampled_patients = rng.choice(patients, size=n_patients, replace=True)

        # 🔹 collect all rows from sampled patients
        df_bs = df[df[patient_col].isin(sampled_patients)]

        y_true_bs = df_bs[y_true_col].values
        y_score_bs = df_bs[y_score_col].values

        # skip if only one class
        if len(np.unique(y_true_bs)) < 2:
            continue

        fpr, tpr, _ = roc_curve(y_true_bs, y_score_bs)
        roc_auc = auc(fpr, tpr)

        # 🔹 interpolate
        interp_tpr = np.interp(mean_fpr, fpr, tpr)
        interp_tpr[0] = 0.0

        tprs.append(interp_tpr)
        aucs.append(roc_auc)

    tprs = np.array(tprs)
    
    mean_tpr = np.mean(tprs, axis=0)
    lower_tpr = np.percentile(tprs, 2.5, axis=0)
    upper_tpr = np.percentile(tprs, 97.5, axis=0)

    lower_auc = np.percentile(aucs, 2.5)
    upper_auc = np.percentile(aucs, 97.5)

    return mean_fpr, mean_tpr, lower_tpr, upper_tpr, full_fpr, full_tpr, roc_auc, lower_auc, upper_auc


def bootstrap_prc_patient_level(
    df,
    y_true_col="label",
    y_score_col="sz_prob",
    patient_col="patient",
    n_bootstrap=1000,
    seed=42
):
    rng = np.random.RandomState(seed)

    prevalence = df[y_true_col].mean()
    patients = df[patient_col].unique()
    n_patients = len(patients)

    mean_recall = np.linspace(0, 1, 200)
    y_true = df[y_true_col].values
    y_score = df[y_score_col].values
    full_precision, full_recall, _ = precision_recall_curve(y_true, y_score)
    # Downsample to 200 values by subsampling (not interpolation)
    if len(full_recall) > 1000:
        indices = np.round(np.linspace(0, len(full_recall) - 1, 1000)).astype(int)
        full_precision = full_precision[indices]
        full_recall = full_recall[indices]
    pr_auc = auc(full_recall, full_precision)

    precisions = []
    aucs = []

    for _ in range(n_bootstrap):
        # 🔹 resample patients
        sampled_patients = rng.choice(patients, size=n_patients, replace=True)

        df_bs = df[df[patient_col].isin(sampled_patients)]

        y_true_bs = df_bs[y_true_col].values
        y_score_bs = df_bs[y_score_col].values

        if len(np.unique(y_true_bs)) < 2:
            continue

        precision, recall, _ = precision_recall_curve(y_true_bs, y_score_bs)

        # sklearn returns recall decreasing → fix ordering
        recall = recall[::-1]
        precision = precision[::-1]

        pr_auc = auc(recall, precision)

        # 🔹 interpolate precision over recall grid
        interp_precision = np.interp(mean_recall, recall, precision)

        # edge handling
        interp_precision[0] = precision[0]

        precisions.append(interp_precision)
        aucs.append(pr_auc)

    precisions = np.array(precisions)

    mean_precision = precisions.mean(axis=0)
    lower_precision = np.percentile(precisions, 2.5, axis=0)
    upper_precision = np.percentile(precisions, 97.5, axis=0)

    mean_auc = np.mean(aucs)
    lower_auc = np.percentile(aucs, 2.5)
    upper_auc = np.percentile(aucs, 97.5)

    return (
        mean_recall,
        mean_precision,
        lower_precision,
        upper_precision,
        full_precision,
        full_recall,
        pr_auc,
        lower_auc,
        upper_auc,
        prevalence
    )

def generate_curve_data(all_data):
    curve_data = {}
    for _, (model, model_data) in enumerate(all_data.items()):
        montages = ordered_montages(model_data, SELECTED_MONTAGES)
        model_curve_data = {'roc':{},'prc':{}}
        for _, mont in enumerate(montages):
            sub = model_data[model_data['montage']==mont]
            model_curve_data['roc'][mont] = bootstrap_roc_patient_level(sub)
            model_curve_data['prc'][mont] = bootstrap_prc_patient_level(sub)
        curve_data[model] = model_curve_data
    return curve_data

def _render_roc_plot(curve_data):
    n_models = len(curve_data)
    fig, axes = plt.subplots(1,n_models,figsize=(4*n_models, 4))
    if n_models == 1:
        axes = [axes]
    for row_ind, (model, model_data) in enumerate(curve_data.items()):
        ax = axes[row_ind]
        montages = SELECTED_MONTAGES
        mont_labels = [label(m) for m in montages]
        for mont_ind, (mont, mont_data) in enumerate(model_data['roc'].items()):
            mean_fpr, mean_tpr, lower_tpr, upper_tpr, full_fpr, full_tpr, mean_auc, lower_auc, upper_auc = mont_data
            ax.plot(full_fpr, full_tpr, color=MONTAGE_PALETTE[mont],
                     label=f"{mont_labels[mont_ind]}, AUC = {mean_auc:.3f} [{lower_auc:.3f}, {upper_auc:.3f}]")
            ax.fill_between(mean_fpr, lower_tpr, upper_tpr, color=MONTAGE_PALETTE[mont], alpha=0.3)
            
        ax.plot([0, 1], [0, 1], linestyle='--',color='k', linewidth=1)
        ax.set_xlabel("False Positive Rate")
        if row_ind == 0:
            ax.set_ylabel("True Positive Rate")
        else:
            ax.set_ylabel("")
            ax.set_yticklabels([])
        ax.set_title(model)
        ax.legend(fontsize=7)
    plt.tight_layout()
    save_figure(out_dir, "suppfig2_ROC")
    
def _render_prc_plot(curve_data):
    n_models = len(curve_data)
    fig, axes = plt.subplots(1,n_models,figsize=(4*n_models, 4))
    if n_models == 1:
        axes = [axes]
    for row_ind, (model, model_data) in enumerate(curve_data.items()):
        ax = axes[row_ind]
        montages = SELECTED_MONTAGES
        mont_labels = [label(m) for m in montages]
        for mont_ind, (mont, mont_data) in enumerate(model_data['prc'].items()):
            mean_recall, mean_precision, lower_p, upper_p, full_precision, full_recall, mean_auc, lower_auc, upper_auc, prevalence = mont_data
            ax.plot(full_recall, full_precision, color=MONTAGE_PALETTE[mont],
                     label=f"{mont_labels[mont_ind]}, AUC = {mean_auc:.3f} [{lower_auc:.3f}, {upper_auc:.3f}]")
            ax.fill_between(mean_recall, lower_p, upper_p, color=MONTAGE_PALETTE[mont], alpha=0.3)
            
        ax.hlines(prevalence, 0, 1, linestyles='--',color='k',linewidth = 1,
                   label=f"Baseline = {prevalence:.3f}")
        ax.set_xlabel("Recall")
        if row_ind == 0:
            ax.set_ylabel("Precision")
        else:
            ax.set_ylabel("")
            ax.set_yticklabels([])
        ax.set_title(model)
        ax.legend(fontsize=7)
    plt.tight_layout()
    save_figure(out_dir, "suppfig2_PRC")

        

In [34]:
SELECTED_MONTAGES = [
    "full",
    "uneeg_diag_bilateral_front",
    "uneeg_diag_left_front",
    "uneeg_diag_right_front",
    "epiminder_2",
]
SELECTED_MODELS = ['SPaRCNet','NDD','SVM']
curve_data = generate_curve_data(prob_data)
with open(f'{out_dir}/roc_prc_curve.pkl','wb') as f:
    pickle.dump(curve_data, f)


100%|██████████| 1000/1000 [05:46<00:00,  2.88it/s]


In [119]:
print("\n=== Supp Fig C: ROC and PRC Curves ===")
_render_roc_plot(curve_data)
_render_prc_plot(curve_data)


=== Supp Fig C: ROC and PRC Curves ===
  Saved: manuscript/suppfig2_ROC.png
  Saved: manuscript/suppfig2_ROC.svg
  Saved: manuscript/suppfig2_PRC.png
  Saved: manuscript/suppfig2_PRC.svg


### SuppFig3 Correlation

In [30]:
def generate_corr(df_p, out_dir, suffix):
    """
    Figure 2: 5 selected montages × 3 models × 3 metrics.
    style = 'strip' (jittered dots) or 'violin'
    """
    models = _get_models(df_p, SELECTED_MODELS)
    montages = SELECTED_MONTAGES
    n_models  = len(models)

    fig, axes = plt.subplots(1, n_models,
        figsize=(4 * n_models, 5), squeeze=True
    )

    mont_labels = [label(m) for m in montages]
    colors = ["#082a54", "#368899","#ffffff", "#F47C6D", "#e02b35"]
    cmap = LinearSegmentedColormap.from_list("custom_cmap", colors, N=256)

    
    for row_i, model in enumerate(models):
        sub = df_p[df_p["model"] == model] if "model" in df_p.columns else df_p
        sub = sub.drop(columns=['admission_id','model'])
        sub = sub[montages]
        ax = axes[row_i]
        # corr = np.corrcoef(sub, rowvar=False)
        corr, p_value = spearmanr(sub.values, axis=0)

        mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

        sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap=cmap, 
                    linewidths=0.4, linecolor="white",
                    cbar=False,
                    ax=ax, vmin=-1, vmax=1,
                    cbar_kws={},
                    annot_kws={"size": 12},
                    )
        ax.set_title(f"{model}", fontweight="bold", pad=8)
        ax.set_xlabel("")
        ax.set_xticklabels(mont_labels, rotation=40, ha='right', fontsize=12)
        ax.set_yticklabels(mont_labels, rotation=0, fontsize=12)
        if row_i != 0:
            ax.set_yticklabels([])
            ax.yaxis.set_tick_params(labelleft=False)
        # ax.yaxis.grid(True, linestyle="--", alpha=0.5)
    # 🔑 create colorbar axis
    cax = fig.add_axes([0.99, 0.27, 0.01, 0.57])#[0.985, 0.10, 0.015, 0.55]
    
    # use one of the heatmaps to generate colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap)
    sm.set_clim(-1, 1)
    
    cbar = fig.colorbar(
        sm,
        cax=cax,
        label='Correlation'
    )
    cbar.outline.set_visible(False)
    cbar.ax.tick_params(labelsize=12)
    cbar.set_ticks([-1, 0, 1])
    
    sns.despine()
    plt.tight_layout()
    save_figure(out_dir, f"suppfig3_corr_{suffix}_spearman")


In [ ]:
SELECTED_MONTAGES = [
    "full",
    "uneeg_diag_bilateral_front",
    "uneeg_diag_left_front",
    "uneeg_diag_right_front",
    "epiminder_2",
]
SELECTED_MODELS = ['SPaRCNet','NDD','SVM']
for suffix in ['f1_event','auroc','auprc']:
    pivoted = df_p[df_p['model'].isin(SELECTED_MODELS)&df_p['montage'].isin(SELECTED_MONTAGES)].pivot_table(
        index=["admission_id","model"],   # group by x and y
        columns=["montage"],        # new column headers
        values=suffix,         # values to fill
        aggfunc="first"     # or 'sum', 'mean', etc. depending on your case
    ).reset_index()
    generate_corr(pivoted, out_dir, suffix)

  Saved: manuscript/suppfig3_corr_f1_event_spearman.png
  Saved: manuscript/suppfig3_corr_f1_event_spearman.svg
  Saved: manuscript/suppfig3_corr_auroc_spearman.png
  Saved: manuscript/suppfig3_corr_auroc_spearman.svg
  Saved: manuscript/suppfig3_corr_auprc_spearman.png
  Saved: manuscript/suppfig3_corr_auprc_spearman.svg


In [17]:
# scatter plot
def generate_scatter_plot(pivoted, out_dir):
    n_models = len(SELECTED_MODELS)
    montages = [m for m in SELECTED_MONTAGES if m != 'full']
    mont_labels = [label(m) for m in montages]
    n_montages = len(montages)
    colors = ["#082a54","#e02b35", "#F47C6D", "#368899"]
    fig,axs = plt.subplots(n_models,n_montages,figsize=(3*n_montages,2.5*n_models))
    for row_i, model_name in enumerate(SELECTED_MODELS):
        sub = pivoted[pivoted['model']==model_name]
        for col_i, m in enumerate(SELECTED_MONTAGES[1:]):
            ax = axs[row_i, col_i]
            ax.scatter(sub["full"], sub[m], color = colors[col_i], s=20, alpha = 0.3)
            if col_i != 0:
                ax.set_yticklabels([])
            if row_i < n_models-1:
                ax.set_xticklabels([])
            if row_i == 0:
                ax.set_title(mont_labels[col_i], fontweight="bold", fontsize=12)
            if col_i == 0:
                ax.set_ylabel(model_name, fontweight="bold", fontsize=12)
                
            # plt.ylabel("Reduced montage F1")
            # identity line
            lims = [0, 1]
            ax.plot(lims, lims, linestyle="--", color = 'k',linewidth=1)
    fig.supxlabel("Full montage F1", fontsize=14, fontweight="bold")
    fig.supylabel("Reduced montage F1", fontsize=14, fontweight="bold")
    sns.despine()
    plt.tight_layout()
    save_figure(out_dir, f"suppfig3_scatter")


In [31]:
pivoted = df_p[df_p['model'].isin(SELECTED_MODELS)&df_p['montage'].isin(SELECTED_MONTAGES)].pivot_table(
    index=["admission_id","model"],   # group by x and y
    columns=["montage"],        # new column headers
    values='f1_event',         # values to fill
    aggfunc="first"     # or 'sum', 'mean', etc. depending on your case
).reset_index()
generate_scatter_plot(pivoted, out_dir)

  Saved: manuscript/suppfig3_scatter.png
  Saved: manuscript/suppfig3_scatter.svg


### Statistical Analysis

In [56]:
SELECTED_MONTAGES = [
    "full",
    "uneeg_diag_bilateral_front",
    "uneeg_diag_left_front",
    "uneeg_diag_right_front",
    "epiminder_2",
]
SELECTED_MODELS = ['SPaRCNet','NDD','SVM']
sub = df_p[df_p['montage'].isin(SELECTED_MONTAGES)]
sub = sub[sub['model'].isin(SELECTED_MODELS)]
pivoted = sub.pivot_table(
    index=["admission_id","model"],   # group by x and y
    columns="montage",        # new column headers
    values="f1_event",         # values to fill
    aggfunc="first"     # or 'sum', 'mean', etc. depending on your case
).reset_index()
pivoted = pivoted.sort_values(['model','admission_id'])
pivoted = pivoted.merge(p_info[['admission_id','laterality','location','epilepsy_type']],on='admission_id',how='left') 
pivoted = pivoted[['admission_id','model']+SELECTED_MONTAGES+['laterality','location','epilepsy_type']]
pivoted.to_csv(f'{out_dir}/statistical_data.csv', index=False)

In [57]:
model_stats = (
    sub
    .groupby('model')[['recall_event', 'fp', 'f1_event']]
    .agg(['mean', 'std'])
)
model_stats.to_csv(f'{out_dir}/stats_basic_performance.csv')

In [74]:
def nakagawa_r2_lmm(fit):
    var_fix = np.var(np.dot(fit.model.exog, fit.fe_params))
    var_re = np.sum([fit.cov_re.iloc[i, i] for i in range(fit.cov_re.shape[0])])
    var_resid = fit.scale
    r2_marginal = var_fix / (var_fix + var_re + var_resid)
    r2_conditional = (var_fix + var_re) / (var_fix + var_re + var_resid)
    return {
        "R2_marginal": r2_marginal,
        "R2_conditional": r2_conditional,
    }

def fit_mixed_ml(formula, data, group_col="admission_id"):
    model = smf.mixedlm(formula, data=data, groups=data[group_col])
    return model.fit(reml=False, method="lbfgs")

def type3_wald_table(mixed_res):
    wt = mixed_res.wald_test_terms(skip_single=False)
    tbl = wt.table.reset_index().rename(columns={"index": "term"}).copy()

    rename = {}
    for c in tbl.columns:
        lc = str(c).lower()
        if "stat" in lc:
            rename[c] = "stat"
        elif "pvalue" in lc or lc in {"p", "p-value"}:
            rename[c] = "pvalue"
        elif "df" in lc:
            rename[c] = "df"
    tbl = tbl.rename(columns=rename)

    keep = [c for c in ["term", "stat", "df", "pvalue"] if c in tbl.columns]
    return tbl[keep]

def get_wald_info(model):
    wt = model.wald_test_terms(skip_single=False)
    tbl = wt.table.reset_index().rename(columns={"index": "Term", "df_constraint": "df"})
    tbl = tbl[['Term','df','statistic','pvalue']]
    tbl['statistic'] = tbl['statistic'].apply(lambda x: np.squeeze(x))
    return tbl

def add_multiindex(df, prefix, ncol):
    cols = list(df.columns)
    df.columns = pd.MultiIndex.from_tuples([
        (col, "") if i < ncol else (prefix, col) for i, col in enumerate(cols)
    ])
    return df

#### Figure2 LinearMixed Model

In [59]:
# Convert R lmer models and r2_nakagawa to Python using statsmodels and custom R2 computation
m0 = fit_mixed_ml("f1_event ~ 1", sub)
m_model = fit_mixed_ml("f1_event ~ C(model)", sub)
m_both = fit_mixed_ml("f1_event ~ C(model) + C(montage)", sub)
m_full = fit_mixed_ml("f1_event ~ C(model) * C(montage)", sub)

Term   df          M0                  M1  \
                              statistic pvalue    statistic   
0            Intercept  1.0  4482.43634    0.0  4228.590690   
1             C(model)  NaN         NaN    NaN  1183.706858   
2           C(montage)  NaN         NaN    NaN          NaN   
3  C(model):C(montage)  NaN         NaN    NaN          NaN   

                                    M2                                   M3  \
                   pvalue    statistic                  pvalue    statistic   
0                     0.0  2968.726188                     0.0  2019.686404   
1  9.147907647394302e-258  1191.959951  1.476336351021363e-259   167.254032   
2                     NaN    45.487279  3.1485470962171816e-09     1.950802   
3                     NaN          NaN                     NaN    63.336195   

                           
                   pvalue  
0                     0.0  
1   4.800079442884294e-37  
2      0.7448074471813962  
3  1.0284526943618393e-10

In [113]:
# Get Wald tables for each model (take only model terms, skip intercept row)
wald_m0 = add_multiindex(get_wald_info(m0), "M0", 2)
wald_m1 = add_multiindex(get_wald_info(m_model), "M1", 2)
wald_m2 = add_multiindex(get_wald_info(m_both), "M2", 2)
wald_m3 = add_multiindex(get_wald_info(m_full), "M3", 2)

summary_df = wald_m3.copy()
summary_df = summary_df.merge(wald_m0[['Term','M0']], on='Term', how='left')
summary_df = summary_df.merge(wald_m1[['Term','M1']], on='Term', how='left')
summary_df = summary_df.merge(wald_m2[['Term','M2']], on='Term', how='left')
summary_df = summary_df[['Term','df','M0','M1','M2','M3']]

r0 = nakagawa_r2_lmm(m0)
r_model = nakagawa_r2_lmm(m_model)
r_both = nakagawa_r2_lmm(m_both)
r_full = nakagawa_r2_lmm(m_full)

r2_m0_m, r2_m0_c = r0["R2_marginal"], r0["R2_conditional"]
r2_m1_m, r2_m1_c = r_model["R2_marginal"], r_model["R2_conditional"]
r2_m2_m, r2_m2_c = r_both["R2_marginal"], r_both["R2_conditional"]
r2_m3_m, r2_m3_c = r_full["R2_marginal"], r_full["R2_conditional"]

r2_tbl = pd.DataFrame([
    ["Marginal R²", r2_m0_m, r2_m1_m, r2_m2_m, r2_m3_m],
    ["Conditional R²", r2_m0_c, r2_m1_c, r2_m2_c, r2_m3_c],
    ["Incremental ΔR²", None, r2_m1_m - r2_m0_m, r2_m2_m - r2_m1_m, r2_m3_m - r2_m2_m],
    ["ΔR² (conditional - marginal)", r2_m0_c - r2_m0_m, r2_m1_c - r2_m1_m, r2_m2_c - r2_m2_m, r2_m3_c - r2_m3_m],
], columns=['Metric', 'M0', 'M1', 'M2', 'M3'])

summary_df.to_csv(f"{out_dir}/supptable2_model_montage_anova.csv", index=False)
r2_tbl.to_csv(f"{out_dir}/supptable2_model_montage_r2.csv", index=False)


In [101]:
def plot_var_parts(var_parts):
    # Use reversed color palette as instructed
    colors = ["#368899", "#F47C6D", "#e02b35", "#082a54", "#cecece"]

    
    labels = [
        "Detector×Montage (fixed)",
        "Montage (fixed)",
        "Detector (fixed)",
        "Patient Admission (random)",
        "Residual"
    ]

    fig, ax = plt.subplots(figsize=(16, 2))
    y_pos = [0]
    left = 0
    for i, (v, label) in enumerate(zip(var_parts, labels)):
        ax.barh(
            y_pos,
            [v],
            left=[left],
            label=label,
            color=colors[i]
        )
        left += v

    ax.set_yticks([])
    ax.set_xlim(0, 1)
    ax.set_xlabel("Proportion of Variance Explained", fontsize=12, fontweight="bold")
    ax.legend(
        loc='center left',
        bbox_to_anchor=(1.01, 0.5),
        ncol=1,
        frameon=False
    )
    ax.set_title("Variance Explained in Mixed Model", fontsize=12, fontweight="bold")
    plt.tight_layout()
    sns.despine()
    save_figure(out_dir, f"fig2A_var_explained")

In [ ]:
r0 = r2_m0_m
r_model = r2_m1_m
r_both = r2_m2_m
r_full = r2_m3_m

d_model = r2_m1_m - r2_m0_m
d_montage = r2_m2_m - r2_m1_m
d_interaction = r2_m3_m - r2_m2_m
d_admission = r2_m3_c - r2_m3_m
d_residual = 1 - r2_m3_c

var_parts = [
        d_interaction,  # Detector×Montage (fixed)
        d_montage,      # Montage (fixed)
        d_model,        # Detector (fixed)
        d_admission,    # Patient Admission (random)
        d_residual      # Residual
    ]
print(var_parts)
    
plot_var_parts(var_parts)

  Saved: manuscript/fig2A_var_explained.png
  Saved: manuscript/fig2A_var_explained.svg


#### Figure2 Statistic Analysis

In [106]:
import scipy
friedman_results = []
all_posthoc_results = {}

for model in ['SPaRCNet', 'NDD', 'SVM']:
    print(model)
    tmp = pivoted[pivoted['model'] == model][SELECTED_MONTAGES].values

    # Perform Friedman test across montages for this model
    stat, p = friedmanchisquare(*[tmp[:, i] for i in range(tmp.shape[1])])
    print("Friedman stat:", stat)
    print("p-value:", p)

    # Save friedman test result
    friedman_results.append({'model': model, 'friedman_stat': stat, 'friedman_p': p})

    # Pairwise posthoc tests if significant
    results = None
    if p <= 0.05:
        n_methods = tmp.shape[1]
        # Ensure pairs are in correct order as in SELECTED_MONTAGES
        pairs = [(i, j) for i in range(n_methods) for j in range(i + 1, n_methods)]

        stats = []
        pvals = []
        rvals = []
        pair_names = []
        mean1s = []
        mean2s = []
        mean_diffs = []

        for i, j in pairs:
            # Always (i, j) where i < j, so SELECTED_MONTAGES[i] vs SELECTED_MONTAGES[j] is guaranteed to be in SELECTED_MONTAGES order
            # Manually compute Wilcoxon signed-rank test statistic W like in R,
            # matching the sign sense and effect direction convention
            diff = tmp[:, i] - tmp[:, j]
            # Remove zero-differences
            nonzero = diff != 0
            diff_nz = diff[nonzero]
            n = diff_nz.size

            # Compute means and mean difference
            mean1 = np.mean(tmp[:, i])
            mean2 = np.mean(tmp[:, j])
            mean_diff = mean1 - mean2

            mean1s.append(mean1)
            mean2s.append(mean2)
            mean_diffs.append(mean_diff)

            # Rank the absolute differences
            abs_ranks = scipy.stats.rankdata(np.abs(diff_nz))
            # W+ is sum of ranks where diff is positive, W- is sum where diff is negative
            Wpos = np.sum(abs_ranks[diff_nz > 0])
            Wneg = np.sum(abs_ranks[diff_nz < 0])

            # In R, wilcox.test reports the sum of positive ranks ("V" or "W"). 
            # We'll use W = min(W+,W-) for two-sided p-value, 
            # but to match R's "effect size r" sign, we use the signed value
            stat_w = Wpos  # This matches the default R reporting

            # For the p-value, use scipy's wrapper (for two-sided test)
            # but for true effect size, follow sign conventions: positive r if i>j, negative if i<j
            # We'll use the continuity-corrected version if possible.
            # Compute the standard stats: 
            mean_W = n * (n + 1) / 4
            std_W = np.sqrt(n * (n + 1) * (2 * n + 1) / 24)
            Z = (stat_w - mean_W) / std_W if std_W > 0 else np.nan
            r = Z / np.sqrt(n) if n > 0 else np.nan

            # Now, as in R, r effect size is positive if the first group tends to be greater, negative otherwise
            if np.median(diff_nz) < 0:
                r = -abs(r)
            elif np.median(diff_nz) > 0:
                r = abs(r)
            else:
                r = 0

            # For the p-value, use scipy's wilcoxon, always use the two-sided value
            # Note: discard zeros for stat/p, but pass zeros for N since pval is calculated correctly only on nonzeros
            p_w = wilcoxon(tmp[:, i], tmp[:, j], zero_method="wilcox", alternative="two-sided")[1]

            stats.append(stat_w)
            pvals.append(p_w)
            rvals.append(r)
            pair_names.append(f"{SELECTED_MONTAGES[i]} vs {SELECTED_MONTAGES[j]}")

        reject, pvals_corrected, _, _ = multipletests(pvals, method='fdr_bh')

        results = pd.DataFrame({
            "comparison": pair_names,
            "stat": stats,
            "mean_group1": mean1s,
            "mean_group2": mean2s,
            "mean_diff": mean_diffs,
            "effect_size": rvals,
            "pval": np.round(pvals, 5),
            "pval_fdr_bh": np.round(pvals_corrected, 5),
            "significant": reject
        })

        print(results)

    # Save post-hoc results even if None
    all_posthoc_results[model] = results

# Save Friedman results to CSV
friedman_df = pd.DataFrame(friedman_results)
friedman_df.to_csv(f"{out_dir}/fig2_friedman_test_results.csv", index=False)

# Save all posthoc results to CSVs
for model, df in all_posthoc_results.items():
    if df is not None:
        posthoc_outfile = f"{out_dir}/fig2_posthoc_wilcoxon_{model}.csv"
        df.to_csv(posthoc_outfile, index=False)
        print(f"Wilcoxon posthoc results for {model} saved to {posthoc_outfile}")

SPaRCNet
Friedman stat: 78.60022203034274
p-value: 3.447346126739185e-16
                                          comparison     stat  mean_group1  \
0                 full vs uneeg_diag_bilateral_front  42343.5     0.644713   
1                      full vs uneeg_diag_left_front  34675.0     0.644713   
2                     full vs uneeg_diag_right_front  52188.0     0.644713   
3                                full vs epiminder_2  52274.0     0.644713   
4  uneeg_diag_bilateral_front vs uneeg_diag_left_...  37256.5     0.639787   
5  uneeg_diag_bilateral_front vs uneeg_diag_right...  44601.0     0.639787   
6          uneeg_diag_bilateral_front vs epiminder_2  51475.5     0.639787   
7    uneeg_diag_left_front vs uneeg_diag_right_front  46386.0     0.619665   
8               uneeg_diag_left_front vs epiminder_2  50161.0     0.619665   
9              uneeg_diag_right_front vs epiminder_2  37614.0     0.567937   

   mean_group2  mean_diff  effect_size     pval  pval_fdr_bh  signif

#### Diagnosis Montage Interaction

In [107]:
sub = sub.merge(p_info[['admission_id', 'epilepsy_type', 'laterality', 'location']], on='admission_id', how='left')
sub[['epilepsy_type','laterality','location']] = sub[['epilepsy_type','laterality','location']].astype(str).replace('Nan', 'Unknown')

In [ ]:
type_tables = []
lat_tables = []
region_tables = []
for mod in ["SPaRCNet", "NDD", "SVM"]:
    print(f"\n{'=' * 80}")
    print(f"Model: {mod}")
    print(f"{'=' * 80}")

    model_data = sub[sub["model"] == mod].copy()

    sub_type = model_data[
        model_data["epilepsy_type"].astype(str).isin(["Focal", "Generalized"])
    ].reset_index(drop=True)
    sub_lat = model_data[
        model_data["laterality"].astype(str).isin(["Left", "Right", "Bilateral"])
    ].reset_index(drop=True)
    sub_region = model_data[
        model_data["location"].astype(str).isin(["Temporal", "Frontal"])
    ].reset_index(drop=True)

    # For each design, perform Wald test and store output
    # col "model", "test_type", plus returned wald table cols

    print("\nEpilepsy type counts (sub_type):")
    print(sub_type["epilepsy_type"].value_counts(dropna=False) if not sub_type.empty else "[empty]")

    print("\nLaterality x location counts (sub_lat):")
    print(pd.crosstab(sub_lat["laterality"], sub_lat["location"], dropna=False) if not sub_lat.empty else "[empty]")

    print("\nLaterality x location counts (sub_region):")
    print(pd.crosstab(sub_region["laterality"], sub_region["location"], dropna=False) if not sub_region.empty else "[empty]")

    # epilepsy_type
    try:
        if sub_type.empty:
            print("\n[skip] montage * epilepsy_type")
        else:
            m = fit_mixed_ml("f1_event ~ C(montage) * C(epilepsy_type)", sub_type)
            print("\n--- Type III Wald: montage * epilepsy_type ---")
            tbl = add_multiindex(get_wald_info(m), mod, 2)
            print(tbl)
            type_tables.append(tbl)
    except Exception as e:
        print(f"\n[error] montage * epilepsy_type: {e}")

    # laterality + location
    try:
        if sub_lat.empty:
            print("\n[skip] montage * laterality + location")
        else:
            m = fit_mixed_ml("f1_event ~ C(montage) * C(laterality) + C(location)", sub_lat)
            print("\n--- Type III Wald: montage * laterality + location ---")
            tbl = add_multiindex(get_wald_info(m), mod, 2)
            print(tbl)
            lat_tables.append(tbl)
    except Exception as e:
        print(f"\n[error] montage * laterality + location: {e}")

    # location + laterality
    try:
        if sub_region.empty:
            print("\n[skip] montage * location + laterality")
        else:
            m = fit_mixed_ml("f1_event ~ C(montage) * C(location) + C(laterality)", sub_region)
            print("\n--- Type III Wald: montage * location + laterality ---")
            tbl = add_multiindex(get_wald_info(m), mod, 2)
            print(tbl)
            region_tables.append(tbl)
    except Exception as e:
        print(f"\n[error] montage * location + laterality: {e}")

# Stack all Wald III tables and save to CSV
concat_tables = []
if type_tables:
    type_df = type_tables[0].copy()
    for i in range(1, len(type_tables)):
        type_df = type_df.merge(type_tables[i].drop(columns=['df']), on='Term', how='right')
    concat_tables.append(type_df)
if lat_tables:
    lat_df = lat_tables[0].copy()
    for i in range(1, len(lat_tables)):
        lat_df = lat_df.merge(lat_tables[i].drop(columns=['df']), on='Term', how='right')
    concat_tables.append(lat_df)
if region_tables:
    region_df = region_tables[0].copy()
    for i in range(1, len(region_tables)):
        region_df = region_df.merge(region_tables[i].drop(columns=['df']), on='Term', how='right')
    concat_tables.append(region_df)
# All have the same columns, add model/test_type/order
try:
    wald_all = pd.concat(concat_tables, axis=0, ignore_index=True)
    wald_all.to_csv(f"{out_dir}/supptable6_montage_disease_interaction.csv", index=False)
except Exception as e:
    print(f"\n[error] montage * disease interaction: {e}")

### Feature analysis

In [ ]:
feature_path = 'detailed_windowed_features.csv'
full_feat = pd.read_csv(feature_path)

In [6]:
# get averaged feature values across all channels, during interictal, peri-ictal, ictal for each event
features = list(full_feat.columns)[6:]
SELECTED_FEATURES = features
feat_label = {'variance':'Variance',
              'amplitude_abs_mean':'Amplitude',
              'amplitude_env_mean':'Envelope',
              'line_length':'Line Length',
              'delta_pow':'Delta Power', 
              'theta_pow':'Theta Power', 
              'alpha_pow':'Alpha Power', 
              'sigma_pow':'Sigma Power',
              'beta_pow':'Beta Power', 
              'gamma_pow':'Gamma Power', 
              'avg_sz_dura':'Seizure Duration'}

In [ ]:
def agg_data(group):
    event_id = group.name
    if 'seizure' in event_id:
        sz = group[group['label']==1]
        # sz = sz.groupby(['window_idx','channel'])[features].agg('mean').reset_index()
        sz = sz.groupby('window_idx')[features].agg('mean').mean(axis=0).to_frame().T
        sz['event_id'] = event_id
        sz['label'] = 1
        return sz
    else:
        iic = group.groupby('window_idx')[features].agg('mean').mean(axis=0).to_frame().T
        iic['event_id'] = event_id
        iic['label'] = 0
        return iic
feature_data = full_feat[~full_feat['channel'].isin(['F3-T3','F4-T4'])].groupby('event_id').apply(agg_data).reset_index(drop=True)
feature_data.to_csv(f'{out_dir}/feature_data.csv',index=False)

In [11]:
feature_path = f'{out_dir}/feature_data.csv'
feature_data = pd.read_csv(feature_path)

In [12]:
for col in feature_data.columns:
    if col in ['event_id', 'label']:
        continue
    feature_data[col+'_log'] = np.log10(feature_data[col])
    median = feature_data[col+'_log'].median()
    iqr = feature_data[col+'_log'].quantile(0.75) - feature_data[col+'_log'].quantile(0.25)
    feature_data[col+'_log_norm'] = (feature_data[col+'_log'] - median) / iqr

In [13]:
for col in features:
    full_feat[col+'_log'] = np.log10(full_feat[col])
    median = full_feat[col+'_log'].median()
    iqr = full_feat[col+'_log'].quantile(0.75) - full_feat[col+'_log'].quantile(0.25)
    full_feat[col+'_log_norm'] = (full_feat[col+'_log'] - median) / iqr

In [14]:
df_s['detected_binary'] = df_s['recall_event']
df_s.loc[df_s['recall_event']==0,'detected_binary'] = 0
df_s.loc[df_s['recall_event']>0,'detected_binary'] = 1
df_s['admission_id'] = df_s['event_id'].apply(lambda x:x.split('_')[0])

In [15]:
tmp = df_s[(df_s['model']=='SPaRCNet')&(df_s['montage']=='full')]
feature_data = feature_data.merge(tmp[['event_id','admission_id','avg_sz_dura','detected_binary']], on='event_id',how='left')
feature_data['sub_class'] = feature_data['detected_binary'].map({np.nan:'Interictal', 0:'Missed',1:'Detected'})

In [16]:
full_feat = full_feat.merge(tmp[['event_id','admission_id','detected_binary']], on='event_id',how='left')
full_feat['sub_class'] = full_feat['detected_binary'].map({np.nan:'Interictal', 0:'Missed',1:'Detected'})

#### Table check

In [ ]:
table1 = TableOne(feature_data, columns=['variance', 'amplitude', 'line_length', 'delta_pow', 'theta_pow',
                                        'alpha_pow', 'beta_pow', 'gamma_pow','avg_sz_dura'], 
                  nonnormal=['variance', 'amplitude', 'line_length', 'delta_pow', 'theta_pow',
                                        'alpha_pow', 'beta_pow', 'gamma_pow','avg_sz_dura'],  groupby='label', missing=True, htest_name=False, pval=True, pval_adjust=None, decimals = 3, include_null=True)


In [ ]:
table1 = TableOne(feature_data[feature_data['sub_class'].isin(['Detected','Missed'])], columns=list(feature_data.columns)[:10], 
                  nonnormal=list(feature_data.columns)[:10],  groupby='sub_class', missing=True, htest_name=False, pval=True, pval_adjust=None, decimals = 3, include_null=True)


In [350]:
table1

Grouped by sub_class                                                                                                             
                                          Missing                           Overall                          Detected                           Missed P-Value
n                                                                              1683                              1195                              488        
variance, median [Q1,Q3]                        0        698.106 [349.795,1764.207]        833.326 [463.336,2938.018]        390.218 [167.042,797.486]  <0.001
amplitude, median [Q1,Q3]                       0          146.456 [95.958,299.237]         177.843 [112.320,453.462]         106.638 [69.197,145.287]  <0.001
line_length, median [Q1,Q3]                     0  46605.994 [19714.292,122590.808]  68895.258 [29333.180,190085.844]  19874.319 [10949.762,41001.053]  <0.001
delta_pow, median [Q1,Q3]                       0        884.013 [396.429,2033.497]       1053.180 [524.687,2715.557]       508.159 [176.801,1155.611]  <0.001
theta_pow, median [Q1,Q3]                       0         227.879 [102.405,462.992]         273.639 [146.954,761.273]         106.371 [45.176,242.313]  <0.001
alpha_pow, median [Q1,Q3]                       0           73.743 [36.606,172.183]           96.434 [51.747,374.447]           37.416 [21.271,66.059]  <0.001
beta_pow, median [Q1,Q3]                        0          103.619 [50.372,384.880]          152.207 [57.971,984.049]          60.962 [34.405,112.487]  <0.001
gamma_pow, median [Q1,Q3]                       0           39.304 [10.705,163.487]           58.193 [12.456,432.680]            18.966 [6.795,48.771]  <0.001
avg_sz_dura, median [Q1,Q3]                     0               1.000 [0.633,1.533]               1.167 [0.767,1.700]              0.667 [0.433,0.967]  <0.001

#### Fig4A Boxplots

In [ ]:
def _render_feat_box_plot(feat_data, style):
    features = SELECTED_FEATURES
    n_feats = len(features)
    feat_labels = [feat_label[f] for f in features]
    n_cols = 4#math.ceil(math.sqrt(n_feats))
    n_rows = 2#math.ceil(n_feats / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols,figsize=(3*n_cols, 3.5*n_rows))
    axes = axes.flatten()
    for col_ind, feat in enumerate(features):
        ax = axes[col_ind]
        # if 'sz_dura' not in feat:
        #     feat = feat+'_log'
        if style == 'raincloud':
            pt.RainCloud(
                    x="sub_class", y=feat, data=feat_data, order=['Interictal','Missed','Detected'], hue_order=['Interictal','Missed','Detected'],
                    palette=['#cecece',  "#e02b35", "#082a54"], #sns.color_palette("viridis"),#
                    bw=0.2, width_viol=0.6, ax=ax, orient="v", alpha=0.8, pointplot=False, move=0.25, offset=0.1, jitter=0.12,
                    box_showfliers=False, rain_alpha=0.5
                )
        elif style == 'box':
            sns.boxplot(
                        data=feat_data, x="sub_class", y=feat, order=['Interictal','Missed','Detected'],
                        palette={'Interictal':'#cecece','Missed':"#e02b35",'Detected':"#082a54"},#sns.color_palette("Paired"),#
                        notch=False, showcaps=True,showfliers=False,width=0.5,
                        # boxprops={'edgeco'}
                        medianprops={"color": "black", "linewidth": 2},zorder=0,
                        ax=ax
                    )
                
            for patch in ax.patches:
                color = patch.get_facecolor()
                patch.set_facecolor((*color[:3], 0.5)) # transparent
                # patch.set_edgecolor(color)
                patch.set_linewidth(1)

            sns.stripplot(
                data=feat_data, x="sub_class", y=feat, order=['Interictal','Missed','Detected'],
                palette={'Interictal':'#cecece','Missed':"#e02b35",'Detected':"#082a54"},#sns.color_palette("Paired"),#
                jitter=0.15, size=4, alpha=0.3, zorder=1, ax=ax,
            )
        ax.set_title(f"{feat_labels[col_ind]}", fontweight="bold", pad=8)
        ax.set_ylabel("")
        ax.set_xlabel("")
        ax.set_yscale('log')
        ax.set_xlim([-0.5,2.5])
        ax.set_xticklabels(['Interictal','Missed','Detected'], fontsize=12)
    sns.despine()
    plt.tight_layout()
    save_figure(out_dir, f"SuppFigX_Feature_{style}")

In [ ]:
SELECTED_FEATURES = ['variance',
 'amplitude_abs_mean',
 'line_length',
 'delta_pow',
 'theta_pow',
 'alpha_pow',
 'beta_pow',
 'avg_sz_dura']
 
 _render_feat_box_plot(feature_data, 'box')

#### Fig4A Statistic tests

In [ ]:
#stat test

group0 = feature_data[feature_data['sub_class'] == 'Interictal']
group1 = feature_data[feature_data['sub_class'] == 'Missed']
group2 = feature_data[feature_data['sub_class'] == 'Detected']

results = []

for f in SELECTED_FEATURES+['avg_sz_dura']:
    x = group0[f].dropna()
    y = group1[f].dropna()
    z = group2[f].dropna()
    
    stat, p = mannwhitneyu(x, y, alternative='two-sided')
    results.append({'Feature':feat_label[f],'Group1':'Interictal','Group2':'Missed','Stats':stat,'P-value':p})
    stat, p = mannwhitneyu(y, z, alternative='two-sided')
    results.append({'Feature':feat_label[f],'Group1':'Missed','Group2':'Detected','Stats':stat,'P-value':p})
results = pd.DataFrame(results).dropna(subset=['P-value'])

In [ ]:
reject, pvals_corrected, _, _ = multipletests(
    results['P-value'],
    alpha=0.05,
    method='fdr_bh'
)
results['Adjusted P'] = pvals_corrected
results.to_csv(f'{out_dir}/fig4_feat_comp_stats.csv',index=False)

#### Fig4B Feature Importance

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import cross_val_predict
from sklearn.inspection import permutation_importance


RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [19]:
# test how features predict detected versus missed
sz_data = feature_data[feature_data['label']==1]
use_cols = [f+'_log_norm' for f in SELECTED_FEATURES]+['avg_sz_dura']
X = sz_data[use_cols].values
y = (1-sz_data["detected_binary"]).astype(int).values  # binary

# If you have multiple seizures per patient, split by patient to avoid leakage:
groups = sz_data["admission_id"].values

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

clf = LogisticRegression(
        penalty="l2",
        solver="lbfgs",
        max_iter=2000,
        class_weight="balanced",  # if missed/detected counts differ a lot
    )

scores = cross_val_score(clf, X, y, cv=cv.split(X, y, groups), scoring="roc_auc")
print("CV ROC-AUC:", scores.mean(), "+/-", scores.std())

clf.fit(X, y)
coef = clf.coef_.ravel()
for name, c in zip(use_cols, coef):
    print(f"{name}: coef (on standardized scale) = {c:.4f}")

CV ROC-AUC: 0.8187078309091482 +/- 0.02070291249652309
amplitude_env_mean_log_norm: coef (on standardized scale) = -1.6042
line_length_log_norm: coef (on standardized scale) = -1.7806
delta_pow_log_norm: coef (on standardized scale) = 1.8381
theta_pow_log_norm: coef (on standardized scale) = -0.9821
alpha_pow_log_norm: coef (on standardized scale) = -0.6703
sigma_pow_log_norm: coef (on standardized scale) = -0.6368
beta_pow_log_norm: coef (on standardized scale) = 1.7021
gamma_pow_log_norm: coef (on standardized scale) = 0.3447
avg_sz_dura: coef (on standardized scale) = -0.4800


In [ ]:

fold_means = []
fold_stds = []
for train_idx, val_idx in cv.split(X, y, groups):
    X_tr, y_tr = X[train_idx], y[train_idx]
    X_va, y_va = X[val_idx], y[val_idx]
    clf.fit(X_tr, y_tr)
    r = permutation_importance(
        clf,
        X_va,
        y_va,
        scoring="roc_auc",
        n_repeats=30,
        random_state=0,
        n_jobs=-1,
    )
    fold_means.append(r.importances_mean)
    fold_stds.append(r.importances_std)
# Aggregate across folds (mean of fold-wise means is the usual summary)
imp_mean_across_folds = np.mean(fold_means, axis=0)
imp_se_across_folds = np.std(fold_means, axis=0, ddof=1) / np.sqrt(len(fold_means))
out = (
    pd.DataFrame({
        "feature": use_cols,
        "importance_mean": imp_mean_across_folds,
        "se_across_folds": imp_se_across_folds,
    })
    .sort_values("importance_mean", ascending=False)
)
out.to_csv(f'{out_dir}/fig4_feature_importance.csv', index=False)

                       feature  importance_mean  se_across_folds
1         line_length_log_norm         0.201109         0.025196
0  amplitude_env_mean_log_norm         0.143588         0.017227
4           alpha_pow_log_norm         0.082886         0.037216
3           theta_pow_log_norm         0.082384         0.014880
2           delta_pow_log_norm         0.079808         0.009726
6            beta_pow_log_norm         0.064702         0.007757
5           sigma_pow_log_norm         0.058692         0.021425
8                  avg_sz_dura         0.042303         0.004507
7           gamma_pow_log_norm         0.012030         0.005545


In [ ]:
plt.barh([feat_label[f.replace('_log_norm','')] for f in out['feature'].to_list()], out['importance_mean'].to_list(), xerr=out['se_across_folds'].to_list(), capsize=5, color='#368899')
plt.xlabel('Feature Importance', fontweight='bold')
plt.yticks(ha='right')
sns.despine()
plt.tight_layout()
# plt.show()
save_figure(out_dir, f"fig4_importance")

  Saved: manuscript/f1_ver_update/Fig4_importance.png
  Saved: manuscript/f1_ver_update/Fig4_importance.svg


'manuscript/f1_ver_update/Fig4_importance.png'

#### Fig4C ROC/PRC plots

In [ ]:
use_cols = [f+'_log_norm' for f in SELECTED_FEATURES]
X = full_feat[use_cols].values
y = full_feat["label"].astype(int).values  # binary

# If you have multiple seizures per patient, split by patient to avoid leakage:
groups = full_feat["admission_id"].values

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

clf = LogisticRegression(
        penalty="l2",
        solver="lbfgs",
        max_iter=2000,
        class_weight="balanced",  # if missed/detected counts differ a lot
    )

pred_prob = cross_val_predict(clf, X, y, cv=cv.split(X, y, groups),method="predict_proba")[:, 1]
full_feat['sz_prob'] = pred_prob

In [ ]:
def _render_feat_prob_roc_prc_plot(full_feat):
    feat = 'sz_prob'
    feat_label = 'Predicted Probability'
    fig, axes = plt.subplots(1, 2, figsize=(4*2, 4))
    ax_roc = axes[0]
    ax_prc = axes[1]
    tmp = full_feat[full_feat['sub_class'].isin(['Interictal', 'Detected'])]
    fpr, tpr, _ = roc_curve(tmp['label'].values, tmp[feat].values)
    auroc_det = auc(fpr,tpr)
    ax_roc.plot(fpr, tpr, color="#082a54",# "#e02b35",
             label=f"Detected, AUROC = {auroc_det:.3f}")
    perc, recall, _ = precision_recall_curve(tmp['label'].values, tmp[feat].values)
    auprc_det = auc(recall, perc)
    ax_prc.plot(recall, perc, color="#082a54",# "#e02b35",
             label=f"Detected, AUPRC = {auprc_det:.3f}")
    prev = np.sum(tmp['label'])/tmp.shape[0]
    ax_prc.hlines(prev, 0, 1, linestyles='--',color='k',linewidth = 1,
       label=f"Detected Baseline = {prev:.3f}")
    # curves for missed
    tmp = full_feat[full_feat['sub_class'].isin(['Interictal', 'Missed'])]
    fpr, tpr, _ = roc_curve(tmp['label'].values, tmp[feat].values)
    auroc_miss = auc(fpr,tpr)
    ax_roc.plot(fpr, tpr, color="#e02b35",
             label=f"Missed, AUROC = {auroc_miss:.3f}")
    perc, recall, _ = precision_recall_curve(tmp['label'].values, tmp[feat].values)
    auprc_miss = auc(recall, perc)
    ax_prc.plot(recall, perc, color="#e02b35",
             label=f"Missed, AUPRC = {auprc_miss:.3f}")
    ax_roc.plot([0, 1], [0, 1], linestyle='--',color='k', linewidth=1)
    prev = np.sum(tmp['label'])/tmp.shape[0]
    ax_prc.hlines(prev, 0, 1, linestyles='--',color='k',linewidth = 1,
       label=f"Missed Baseline = {prev:.3f}")
    ax_roc.set_xlabel("False Positive Rate",fontweight='bold')
    ax_prc.set_xlabel("Recall",fontweight='bold')
    ax_roc.set_ylabel("True Positive Rate",fontweight='bold')
    ax_prc.set_ylabel("Precision",fontweight='bold')
    ax_roc.set_title('ROC Curve',fontweight='bold')
    ax_prc.set_title('Precision-recall Curve',fontweight='bold')
    ax_roc.legend(fontsize=8)
    ax_prc.legend(fontsize=8)
    sns.despine()
    plt.tight_layout()
    save_figure(out_dir, f"fig4C_feature_roc_prc")

In [512]:
_render_feat_prob_roc_prc_plot(full_feat)

  Saved: manuscript/f1_ver_update/SuppFigX_Feature_Prob_AUC_windowlevel.png
  Saved: manuscript/f1_ver_update/SuppFigX_Feature_Prob_AUC_windowlevel.svg
